## 1. Nastavení

In [5]:
DRIVE_DIR = "/content/drive/MyDrive/dna-lm/data/raw/genome"   # cílová složka na Drive
WORK_DIR = "/content/genome"                                   # lokální disk Colabu (rychlý)

BASE_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/analysisSet"
FA_GZ = "hg38.analysisSet.fa.gz"
OUT_FA = "hg38.main.fa"

# chr1–22, X, Y a mitochondrie. Výběr pro trénink (1–22 + X) se dělá až při generování oken.
KEEP = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"]

## 2. Připojení Google Drive

Ve VS Code s Colab rozšířením případně použij příkaz `Colab: Mount Google Drive to Server...`.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
!df -h /content | tail -1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
overlay         108G   22G   87G  20% /


## 3. Pomocné funkce

In [7]:
import gzip
import hashlib
import json
import os
import shutil
import time
import urllib.request
from datetime import datetime, timezone

from tqdm.auto import tqdm


def download(url, dest):
    """Stáhne soubor s progress barem; pokud už existuje celý, přeskočí."""
    with urllib.request.urlopen(url) as r:
        total = int(r.headers.get("Content-Length", 0))
        if os.path.exists(dest) and os.path.getsize(dest) == total:
            print(f"Už staženo: {dest}")
            return
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                         desc=os.path.basename(dest)) as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))


def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 24):
            h.update(chunk)
    return h.hexdigest()


def expected_md5(md5_url, filename):
    text = urllib.request.urlopen(md5_url).read().decode()
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 2 and parts[1].lstrip("*") == filename:
            return parts[0]
    raise ValueError(f"{filename} není v {md5_url}")


def filter_fasta(src_gz, dst_fa, keep):
    """
    Projde gzip FASTA jedním průchodem, ponechá jen chromozomy z `keep`,
    zapíše nekomprimovanou FASTA, vytvoří k ní .fai index a vrátí statistiky.
    """
    keep = set(keep)
    stats, fai = [], []
    cur = None
    offset = 0

    def close(c):
        if c is not None:
            stats.append({k: c[k] for k in ("chrom", "length", "n_count", "softmasked")})
            fai.append(f"{c['chrom']}\t{c['length']}\t{c['offset']}\t{c['lb']}\t{c['lb'] + 1}")

    with gzip.open(src_gz, "rt") as fin, open(dst_fa, "w") as fout:
        writing = False
        for line in tqdm(fin, desc="Filtrování", unit=" řádků", mininterval=5):
            if line.startswith(">"):
                close(cur)
                cur = None
                name = line[1:].split()[0]
                writing = name in keep
                if writing:
                    header = f">{name}\n"
                    fout.write(header)
                    offset += len(header)
                    cur = {"chrom": name, "length": 0, "n_count": 0, "softmasked": 0,
                           "offset": offset, "lb": None, "short_seen": False}
                continue
            if not writing:
                continue
            seq = line.rstrip("\n")
            n = len(seq)
            if cur["lb"] is None:
                cur["lb"] = n
            elif cur["short_seen"] or n > cur["lb"]:
                raise ValueError(f"Nestejná délka řádků v {cur['chrom']}, .fai by byl špatně")
            if n < cur["lb"]:
                cur["short_seen"] = True
            cur["length"] += n
            cur["n_count"] += seq.count("N") + seq.count("n")
            cur["softmasked"] += sum(seq.count(b) for b in "acgtn")
            fout.write(seq + "\n")
            offset += n + 1
        close(cur)

    with open(dst_fa + ".fai", "w") as f:
        f.write("\n".join(fai) + "\n")
    missing = keep - {s["chrom"] for s in stats}
    if missing:
        raise ValueError(f"Chybí chromozomy: {sorted(missing)}")
    return stats

## 4. Stažení a kontrola MD5

In [8]:
gz_path = os.path.join(WORK_DIR, FA_GZ)
download(f"{BASE_URL}/{FA_GZ}", gz_path)

md5_expected = expected_md5(f"{BASE_URL}/md5sum.txt", FA_GZ)
md5_actual = md5sum(gz_path)
print("očekávané MD5:", md5_expected)
print("spočítané MD5:", md5_actual)
assert md5_actual == md5_expected, "MD5 nesedí, soubor smaž a stáhni znovu"

Už staženo: /content/genome/hg38.analysisSet.fa.gz
očekávané MD5: 6d3c82e1e12b127d526395294526b9c8
spočítané MD5: 6d3c82e1e12b127d526395294526b9c8


## 5. Filtrování chromozomů, index a statistiky

In [9]:
import pandas as pd

fa_path = os.path.join(WORK_DIR, OUT_FA)
t0 = time.time()
stats = filter_fasta(gz_path, fa_path, KEEP)
print(f"Hotovo za {time.time() - t0:.0f} s")

df = pd.DataFrame(stats)
df["n_frac"] = df["n_count"] / df["length"]
df["softmasked_frac"] = df["softmasked"] / df["length"]
df = df.set_index("chrom").loc[KEEP]
print(f"Celkem: {df['length'].sum():,} bp, N: {df['n_count'].sum():,}, "
      f"soft-masked: {df['softmasked'].sum() / df['length'].sum():.1%}")
df

Filtrování: 0 řádků [00:00, ? řádků/s]

Hotovo za 220 s
Celkem: 3,088,286,401 bp, N: 164,553,847, soft-masked: 49.6%


,length,n_count,softmasked,n_frac,softmasked_frac
chrom,,,,,
chr1,248956422,18475410,119210542,0.074211,0.478841
chr2,242193529,1645301,118388569,0.006793,0.488818
chr3,198295559,195424,101697004,0.000986,0.512856
chr4,190214555,461888,98155263,0.002428,0.516024
chr5,181538259,2555066,93295241,0.014075,0.513915
chr6,170805979,727457,85112958,0.004259,0.498302
chr7,159345973,375842,81632871,0.002359,0.512300
chr8,145138636,370500,74291317,0.002553,0.511865
chr9,138394717,16604167,62959077,0.119977,0.454924


## 6. Uložení na Google Drive

In [10]:
meta = {
    "source": f"{BASE_URL}/{FA_GZ}",
    "assembly": "GRCh38 (UCSC hg38 analysis set, no alt contigs)",
    "md5_fa_gz": md5_actual,
    "downloaded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "chromosomes": KEEP,
    "total_bp": int(df["length"].sum()),
    "softmasking": "lowercase = RepeatMasker + Tandem Repeats Finder",
}
with open(os.path.join(WORK_DIR, "metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)
df.to_csv(os.path.join(WORK_DIR, "chrom_stats.csv"))

for name in [FA_GZ, OUT_FA, OUT_FA + ".fai", "metadata.json", "chrom_stats.csv"]:
    src = os.path.join(WORK_DIR, name)
    dst = os.path.join(DRIVE_DIR, name)
    print(f"Kopíruji {name} ({os.path.getsize(src) / 1e9:.2f} GB)...")
    shutil.copy(src, dst)
!ls -lh {DRIVE_DIR}

Kopíruji hg38.analysisSet.fa.gz (0.95 GB)...
Kopíruji hg38.main.fa (3.15 GB)...
Kopíruji hg38.main.fa.fai (0.00 GB)...
Kopíruji metadata.json (0.00 GB)...
Kopíruji chrom_stats.csv (0.00 GB)...
total 3.9G
-rw------- 1 root root 1.8K Oct 19 11:12 chrom_stats.csv
-rw------- 1 root root 906M Oct 19 11:12 hg38.analysisSet.fa.gz
-rw------- 1 root root 3.0G Oct 19 11:12 hg38.main.fa
-rw------- 1 root root  788 Oct 19 11:12 hg38.main.fa.fai
-rw------- 1 root root  704 Oct 19 11:12 metadata.json


## 7. Kontrola: čtení z Drive přes pyfaidx

In [11]:
!pip install -q pyfaidx
from pyfaidx import Fasta

genome = Fasta(os.path.join(DRIVE_DIR, OUT_FA))   # použije uložený .fai
print(list(genome.keys()))
print(len(genome["chr1"]), "bp na chr1")
print(genome["chr1"][1_000_000:1_000_120])        # 120 bp z chr1

['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX', 'chrY', 'chrM']
248956422 bp na chr1
GTGGAGCGCGCCGCCACGGACCAcgggcgggctggcgggcgagcggcgagcgcgcggCGATCCGAGCCCCTAGGGCGGATCCCGGCTCCAGGCCCGCGCGCGCCTCAGGCCGTTTCCCTA


Hotovo. Do `data/README.md` v repu si poznamenej zdroj, MD5 a datum stažení
(jsou v `metadata.json` na Drive), budou potřeba do kapitoly Data.